# Estimator Sheet 
This notebook builds the single combined dataset used for quantile regression estimation. It walks a folder of forecast-card CSVs (`project-*.csv`, `poi-*.csv`, `observations-*.csv`), finds every project, combines each project's files into one merged table, adds K/D factors, peak-hour volume, and post-construction capacity (via the Post-Construction Capacity Expansion Equation), and writes everything out to a single `combined_output.csv`. That combined file is the input for the model-fitting notebook.

**How to use this notebook:**
1. Run the "find project folders" cell to confirm every project was located. Set `DATA_ROOT` to the folder containing all your project CSVs.
2. Run the "build the combined dataset" cell. For each project this loads its project, poi, and observations CSVs, merges them together, and labels each observation as `before` or `after` the project's actual open date. All projects are then stacked into one `master_df`.
3. Run the "years from open" cell to add a `years_from_open` column to `master_df`, the difference between each observation's year and the project's actual open year.
4. Run the "K/D factors, peak-hour volume, and post-construction capacity" cells to add `k_factor`, `d_factor`, `peak_hour_volume`, `engineering_capacity_ceng`, `capacity_multiplier`, and `peak_hour_capacity` to `master_df`. `peak_hour_capacity` is now computed with the **Post-Construction Capacity Expansion Equation** (SPR20-597-consistent engineering baseline x an exponential multiplier built from the calibration betas in `Cap_Cal_Values.csv`), instead of the old flat HCM-style lookup.
5. Run the "spot-check" cell to visually confirm the merge and factor calculations worked as expected.
6. Run the "write out" cell to save `master_df` to `combined_output.csv`.

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1. Find your project folders

Point `DATA_ROOT` at the folder holding your project data. This walks the entire directory tree underneath it and collects every sub-folder that contains a `project-*.csv` file (each matching folder is treated as one project). The result is a sorted list of folder paths, printed below so you can confirm the right projects were found.

**Update this path to point at your own data folder before continuing.**

In [2]:
def find_project_folders(root_dir):
    """Return every sub-folder of root_dir that contains a project-*.csv file."""
    folders = []
    for dirpath, _dirnames, filenames in os.walk(root_dir):
        if any(re.match(r"project-.*\.csv$", f, re.IGNORECASE) for f in filenames):
            folders.append(dirpath)
    return sorted(folders)

DATA_ROOT = r"D:\Documents\Code\forecastcards_ba_data\data"
project_folders = find_project_folders(DATA_ROOT)
project_folders

['D:\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-0436',
 'D:\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-0507',
 'D:\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9310',
 'D:\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9503',
 'D:\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9506',
 'D:\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9508',
 'D:\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9603',
 'D:\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9605',
 'D:\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9607',
 'D:\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9608',
 'D:\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9702UPDATE',
 'D:\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9706',
 'D:\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9707',
 'D:\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9712F',
 'D:\\Documents\\Code\\forecastcards_ba_data\\data\\MN_F-M-9801',
 'D

## 2. Build the combined dataset

This section does the actual finding, loading, and merging for every project, then stacks the results into one master table used for estimation.

**Loading a project's files** — for a given project folder, finds and reads:
- `project-*.csv` → project-level metadata (one row per project)
- `poi-*.csv` → "points of interest" tied to that project (e.g. intersections/segments)
- `observations-*.csv` → observed traffic/volume records

**Merging a project's files** — performs two sequential merges:
- **project → poi:** tags each POI row with its `project_id`, then merges project metadata onto every POI row (keeping all POIs, broadcasting the single project row across them).
- **before/after labeling:** converts `observation_date` to a real datetime and compares it to the project's `date_open_actual` to create `before_or_after`.
- **(project+poi) → observations:** merges the project/poi table onto the observations, matching on `poi_id` **and** `before_or_after`, keeping every observation row, and renames `start_time`/`end_time` to the `_observation` versions for consistent naming.


In [3]:
def load_project_files(folder):
    """Load the project / poi / observations csvs for one folder."""
    project_path = glob.glob(os.path.join(folder, "project-*.csv"))[0]
    poi_path = glob.glob(os.path.join(folder, "poi-*.csv"))[0]
    obs_path = glob.glob(os.path.join(folder, "observations-*.csv"))[0]

    project_df = pd.read_csv(project_path)
    poi_df = pd.read_csv(poi_path)
    obs_df = pd.read_csv(obs_path)

    return project_df, poi_df, obs_df

In [4]:
def process_project_folder(folder):
    project_df, poi_df, obs_df = load_project_files(folder)

    project_id = project_df["project_id"].iloc[0]
    date_open_actual = pd.to_datetime(project_df["date_open_actual"].iloc[0])

    poi_df = poi_df.copy()

    if "before_or_after" in poi_df.columns:
        poi_df["before_or_after"] = (
            poi_df["before_or_after"].astype("string").str.strip().str.lower()
        )
    poi_df["project_id"] = project_id

    # STEP 1: project -> poi
    project_poi = pd.merge(project_df, poi_df, on="project_id", how="right")

    # derive a sensible before/after label for each observation
    obs_df = obs_df.copy()
    obs_df["observation_date"] = pd.to_datetime(obs_df["observation_date"], errors="coerce")
    obs_df["before_or_after"] = pd.array(
        np.where(obs_df["observation_date"] < date_open_actual, "before", "after"),
        dtype="string",
    )
    obs_df["before_or_after_binary"] = (obs_df["before_or_after"] == "after").astype(int)

    # STEP 2: (project + poi) -> observation
    merged = pd.merge(
        project_poi, obs_df, on=["poi_id", "before_or_after"], how="right"
    )

    merged = merged.rename(columns={"start_time": "start_time_observation",
                                     "end_time": "end_time_observation"})

    return merged

In [5]:
project_tables = [process_project_folder(folder) for folder in project_folders]

master_df = pd.concat(project_tables, ignore_index=True, sort=False)

print(f"{len(master_df)} total observation rows across {len(project_tables)} project(s)")
master_df.head(10)

1964 total observation rows across 62 project(s)


,project_id,state,county,project_type,description,date_open_planned,date_horizon,construction_start_date,date_open_actual,area_type,project_length_miles,poi_id,before_or_after,facility_name,segment_length_miles,speed_limit_mph,lane_width,signals_per_mile,stop_signs_per_mile,through_lanes,left_turn_lanes,right_turn_lanes,express_hov_hot_lanes,shoulder_width,functional_class,obs_id,observation_date,start_time_observation,end_time_observation,observation_variable,observation_value,observation_unit,before_or_after_binary
0,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,unknown,4/1/2017,suburban,0.59,40974,before,Lexington Ave.,0.07,55,12,0,0,2,0,0,0,7,minor arterial,40974_1,2011-01-01,0:00:00,24:00:00,volume,16700.0,AADT,0
1,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,unknown,4/1/2017,suburban,0.59,40974,before,Lexington Ave.,0.07,55,12,0,0,2,0,0,0,7,minor arterial,40974_2,2016-01-01,0:00:00,24:00:00,volume,17518.0,AADT,0
2,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,unknown,4/1/2017,suburban,0.59,40974,before,Lexington Ave.,0.07,55,12,0,0,2,0,0,0,7,minor arterial,40974_3,2017-01-01,0:00:00,24:00:00,volume,17844.0,AADT,0
3,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,unknown,4/1/2017,suburban,0.59,40974,after,Lexington Ave.,0.07,55,12,0,0,2,0,1,0,9,minor arterial,40974_4,2018-01-01,0:00:00,24:00:00,volume,17937.0,AADT,1
4,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,unknown,4/1/2017,suburban,0.59,40974,after,Lexington Ave.,0.07,55,12,0,0,2,0,1,0,9,minor arterial,40974_5,2019-01-01,0:00:00,24:00:00,volume,15200.0,AADT,1
5,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,unknown,4/1/2017,suburban,0.59,40974,after,Lexington Ave.,0.07,55,12,0,0,2,0,1,0,9,minor arterial,40974_6,2020-01-01,0:00:00,24:00:00,volume,11388.0,AADT,1
6,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,unknown,4/1/2017,suburban,0.59,40974,after,Lexington Ave.,0.07,55,12,0,0,2,0,1,0,9,minor arterial,40974_7,2021-01-01,0:00:00,24:00:00,volume,13169.0,AADT,1
7,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,unknown,4/1/2017,suburban,0.59,40974,after,Lexington Ave.,0.07,55,12,0,0,2,0,1,0,9,minor arterial,40974_8,2022-01-01,0:00:00,24:00:00,volume,13525.0,AADT,1
8,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,unknown,4/1/2017,suburban,0.59,40974,after,Lexington Ave.,0.07,55,12,0,0,2,0,1,0,9,minor arterial,40974_9,2023-01-01,0:00:00,24:00:00,volume,12691.0,AADT,1
9,F-M-0436,MN,Ramsey,road-capacity-expansion,Replace Bridge over Lexington Ave and remove W...,1/1/2010,1/1/2080,unknown,4/1/2017,suburban,0.59,40974,after,Lexington Ave.,0.07,55,12,0,0,2,0,1,0,9,minor arterial,40974_10,2024-01-01,0:00:00,24:00:00,volume,13230.0,AADT,1


## 3. Years from open (before/after completion year)

Adds a `years_from_open` column to `master_df`: the difference between the calendar year of each observation and the calendar year the project actually opened (`date_open_actual`).

In [6]:
master_df["observation_date"] = pd.to_datetime(master_df["observation_date"], errors="coerce")
master_df["date_open_actual"] = pd.to_datetime(master_df["date_open_actual"], errors="coerce")

master_df["years_from_open"] = (
    master_df["observation_date"].dt.year - master_df["date_open_actual"].dt.year
)

master_df[["project_id", "poi_id", "observation_date", "date_open_actual", "years_from_open"]].head(10)

,project_id,poi_id,observation_date,date_open_actual,years_from_open
0,F-M-0436,40974,2011-01-01,2017-04-01,-6.0
1,F-M-0436,40974,2016-01-01,2017-04-01,-1.0
2,F-M-0436,40974,2017-01-01,2017-04-01,0.0
3,F-M-0436,40974,2018-01-01,2017-04-01,1.0
4,F-M-0436,40974,2019-01-01,2017-04-01,2.0
5,F-M-0436,40974,2020-01-01,2017-04-01,3.0
6,F-M-0436,40974,2021-01-01,2017-04-01,4.0
7,F-M-0436,40974,2022-01-01,2017-04-01,5.0
8,F-M-0436,40974,2023-01-01,2017-04-01,6.0
9,F-M-0436,40974,2024-01-01,2017-04-01,7.0


## 4. K/D factors and peak-hour volume

Adds `k_factor`, `d_factor`, and `peak_hour_volume` using `K_D_Factors.csv`, matched to each row's `functional_class`:

- `k_factor`, `d_factor` — looked up directly from `K_D_Factors.csv`.
- `peak_hour_volume` — `AADT x K x D`, computed from the observed volume column (only where the unit is `AADT`).

Rows with `functional_class == "unknown"` will come out blank for these columns.

(`Cap_Base_Values.csv` and the old flat HCM lookup are no longer used for capacity — see the next section.)

In [7]:
# Paths to the factor tables
FACTOR_TABLES_ROOT = r"D:\Documents\Code\forecastcards_v2\forecastcards\capacity_factor_tables"

K_D_FACTORS_PATH = os.path.join(FACTOR_TABLES_ROOT, "K_D_Factors.csv")
CAP_CAL_PATH = os.path.join(FACTOR_TABLES_ROOT, "Cap_Cal_Values.csv")

def normalize_functional_class(value):
    """Lowercase/trim a functional class label so it matches across files
    (e.g. K_D_Factors.csv uses 'Local', Cap_Base_Values.csv uses 'Local Road')."""
    if pd.isna(value):
        return None
    label = str(value).strip().lower()
    if label == "local road":
        label = "local"
    return label

# K / D factors by functional class
kd_factors = pd.read_csv(K_D_FACTORS_PATH)[["Functional Class", "K Factor", "D Factor"]].dropna(subset=["Functional Class"])
kd_factors["_fc"] = kd_factors["Functional Class"].apply(normalize_functional_class)
k_factor_lookup = kd_factors.set_index("_fc")["K Factor"].to_dict()
d_factor_lookup = kd_factors.set_index("_fc")["D Factor"].to_dict()

master_df["through_lanes"] = pd.to_numeric(master_df["through_lanes"], errors="coerce")

_fc_norm = master_df["functional_class"].apply(normalize_functional_class)
master_df["k_factor"] = _fc_norm.map(k_factor_lookup)
master_df["d_factor"] = _fc_norm.map(d_factor_lookup)

# Peak hour volume = AADT x K x D, for the observed volume column
master_df["peak_hour_volume"] = np.where(
    master_df["observation_unit"] == "AADT",
    master_df["observation_value"] * master_df["k_factor"] * master_df["d_factor"],
    np.nan,
)

master_df[[
    "functional_class", "through_lanes", "k_factor", "d_factor",
    "observation_value", "peak_hour_volume",
]].head(10)

,functional_class,through_lanes,k_factor,d_factor,observation_value,peak_hour_volume
0,minor arterial,2.0,0.1,0.55,16700.0,918.500
1,minor arterial,2.0,0.1,0.55,17518.0,963.490
2,minor arterial,2.0,0.1,0.55,17844.0,981.420
3,minor arterial,2.0,0.1,0.55,17937.0,986.535
4,minor arterial,2.0,0.1,0.55,15200.0,836.000
5,minor arterial,2.0,0.1,0.55,11388.0,626.340
6,minor arterial,2.0,0.1,0.55,13169.0,724.295
7,minor arterial,2.0,0.1,0.55,13525.0,743.875
8,minor arterial,2.0,0.1,0.55,12691.0,698.005
9,minor arterial,2.0,0.1,0.55,13230.0,727.650


## 5. Post-Construction Capacity Expansion Equation

Replaces the old flat `Cap_base x N x f_hv x PHF x f_p x f_g` lookup with the full model from
*Proposed Post-Capacity Expansion Equation (rev2)*:

$$\text{PostCapacity}_i = C_{eng,i} \times \exp\big(\beta_{LM}\Delta LM_i + \beta_L \Delta L_i + \beta_{LT}\Delta LT_i + \beta_{RT}\Delta RT_i + \beta_W W_i + \beta_E E_i + \beta_S S_i + \beta_F F_i + \beta_A \Delta\ln(AADT_i) + \beta_P \Delta\ln(Pop_r) + \beta_Y \Delta\ln(Inc_r) + \beta_{SL} SL_i + \beta_{SIG} SIG_i + \beta_{AT} AT_i\big)$$

**$C_{eng}$ (engineering baseline)** is computed with the SPR20-597 HERS free-flow-speed method (Section 3
of the doc): FFS → lane-width penalty → signal adjustment → stop-sign adjustment → speed at capacity
(90% of FFS, per HCM density = 35 pc/mi/ln) → flow per lane → x lanes x heavy-vehicle factor. It uses each
POI's **post-construction** geometry, per the equation's definition of $C_{eng}$.

In [8]:
# Load calibration betas (and HCM scalars) from Cap_Cal_Values.csv.

cap_cal = pd.read_csv(CAP_CAL_PATH, encoding="cp1252")

PARAM_TO_KEY = {
    "Heavy Vehicle Factor": "f_hv",
    "Peak Hour Factor": "PHF",
    "Driver Population Factor": "f_p",
    "Grade Adjustment Factor": "f_g",
    "Lanemiles Elasticity": "bLM",
    "Lane Count Effect": "bL",
    "Left?Turn Lane Effect": "bLT",
    "Right?Turn Lane Effect": "bRT",
    "Lane Width Effect": "bW",
    "Managed Lane Effect": "bE",
    "Shoulder Widening Effect": "bS",
    "Functional Class Effect": "bF",
    "Speed Limit Effect": "bSL",
    "Signals per Mile Effect": "bSIG",
    "Area Type Effect": "bAT",
    "Local Demand Change": "bA",
    "Population Growth": "bP",
    "Income Growth": "bY",
}
cap_cal["_key"] = cap_cal["Parameter"].str.strip().map(PARAM_TO_KEY)
missing = cap_cal[cap_cal["_key"].isna()]
if len(missing):
    print("WARNING: unrecognized rows in Cap_Cal_Values.csv (not used):", missing["Parameter"].tolist())

cap_cal["_value"] = pd.to_numeric(cap_cal["Default Value"], errors="coerce")

bad_parse = cap_cal[cap_cal["_value"].isna()]
if len(bad_parse):
    print("WARNING: could not parse Default Value for:", bad_parse["Parameter"].tolist())

betas = cap_cal.dropna(subset=["_key"]).set_index("_key")["_value"].to_dict()

# Sanity check against the doc's Section 5 priors
for sym in ["bSIG", "bAT"]:
    if sym in betas and betas[sym] > 0:
        print(f"WARNING: {sym} = {betas[sym]} parsed as positive but the doc expects it negative "
              f"(see Proposed_Post-Capacity_Expansion_Equation_rev2, Section 5) - check Cap_Cal_Values.csv.")

f_hv = betas["f_hv"]  # heavy-vehicle factor, used in the Ceng calc below

betas

{'f_hv': 0.95,
 'PHF': 0.92,
 'f_p': 0.95,
 'f_g': 1.0,
 'bLM': 0.45,
 'bL': 0.35,
 'bLT': 0.1,
 'bRT': 0.06,
 'bW': 0.05,
 'bE': 0.15,
 'bS': 0.05,
 'bF': 0.1,
 'bSL': 0.05,
 'bSIG': -0.1,
 'bAT': -0.1,
 'bA': 1.0,
 'bP': 0.25,
 'bY': 0.25}

In [9]:
# Per-POI before/after geometry, used both for Ceng (post-construction geometry)
# and for the Delta/indicator variables in the exponential multiplier.
GEOM_COLS = [
    "through_lanes", "lane_width", "left_turn_lanes", "right_turn_lanes",
    "shoulder_width", "express_hov_hot_lanes", "speed_limit_mph",
    "signals_per_mile", "stop_signs_per_mile", "segment_length_miles",
]
for c in GEOM_COLS:
    master_df[c] = pd.to_numeric(master_df[c], errors="coerce")

geom_by_period = (
    master_df.groupby(["poi_id", "before_or_after"])[GEOM_COLS].mean().unstack("before_or_after")
)

def pre(col):
    return geom_by_period[col]["before"] if "before" in geom_by_period[col] else np.nan

def post(col):
    return geom_by_period[col]["after"] if "after" in geom_by_period[col] else np.nan

# post-construction functional class / area type, used for Ceng flow-type and AT level
_after_rows = master_df[master_df["before_or_after"] == "after"]
fc_post = _after_rows.groupby("poi_id")["functional_class"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else None)
at_post = _after_rows.groupby("poi_id")["area_type"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else None)

poi = pd.DataFrame(index=geom_by_period.index)

# --- geometry deltas (post - pre / pre) ---
lane_miles_pre = pre("through_lanes") * pre("segment_length_miles")
lane_miles_post = post("through_lanes") * post("segment_length_miles")
poi["d_lane_miles"] = (lane_miles_post - lane_miles_pre) / lane_miles_pre
poi["d_through_lanes"] = post("through_lanes") - pre("through_lanes")
poi["d_left_turn_lanes"] = post("left_turn_lanes") - pre("left_turn_lanes")
poi["d_right_turn_lanes"] = post("right_turn_lanes") - pre("right_turn_lanes")

# --- 0/1 indicators (W, E, S, SL) ---
poi["lane_width_improved"] = (post("lane_width") > pre("lane_width")).astype("Int64")
poi["express_lane_added"] = (post("express_hov_hot_lanes") > pre("express_hov_hot_lanes")).astype("Int64")
poi["shoulder_widened"] = (post("shoulder_width") > pre("shoulder_width")).astype("Int64")
poi["speed_limit_increased"] = (post("speed_limit_mph") > pre("speed_limit_mph")).astype("Int64")

# --- functional class improvement (F): no ranking data, so left at 0 unless enriched later ---
poi["functional_class_improved"] = 0

# --- levels used directly (SIG, AT), from post-construction state ---
poi["signals_per_mile_post"] = post("signals_per_mile")
poi["stop_signs_per_mile_post"] = post("stop_signs_per_mile")
poi["through_lanes_post"] = post("through_lanes")
poi["lane_width_post"] = post("lane_width")
poi["speed_limit_post"] = post("speed_limit_mph")
poi["functional_class_post"] = fc_post
poi["area_type_post"] = at_post
poi["area_type_level"] = at_post.astype(str).str.strip().str.lower().isin(["urban", "suburban"]).astype(int)

# --- demand delta: mean AADT pre vs post per POI ---
aadt_obs = master_df[master_df["observation_unit"] == "AADT"]
aadt_by_period = aadt_obs.groupby(["poi_id", "before_or_after"])["observation_value"].mean().unstack("before_or_after")
poi["d_ln_aadt"] = np.log(aadt_by_period["after"] / aadt_by_period["before"])

poi.head(10)

,d_lane_miles,d_through_lanes,d_left_turn_lanes,d_right_turn_lanes,lane_width_improved,express_lane_added,shoulder_widened,speed_limit_increased,functional_class_improved,signals_per_mile_post,stop_signs_per_mile_post,through_lanes_post,lane_width_post,speed_limit_post,functional_class_post,area_type_post,area_type_level,d_ln_aadt
poi_id,,,,,,,,,,,,,,,,,,
6788,0.000000e+00,0.0,0.0,0.0,0,0,0,0,0,1.0,0.0,4.0,12.0,55.0,freeway,rural,0.0,0.009838
6789,0.000000e+00,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,2.0,12.0,55.0,principal arterial,rural,0.0,-0.074592
9834,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,principal arterial,rural,0.0,NaN
9836,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,principal arterial,rural,0.0,NaN
9860,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,principal arterial,rural,0.0,NaN
9873,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,principal arterial,rural,0.0,NaN
9884,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,freeway,suburban,1.0,NaN
10026,-1.423363e-16,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,2.0,11.0,55.0,minor arterial,rural,0.0,0.117709
10027,0.000000e+00,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,2.0,12.0,55.0,minor arterial,rural,0.0,-0.085724


In [10]:
# SPR20-597 HERS free-flow-speed method -> engineering baseline capacity (Ceng), Sections 2-3 of the doc.
# Uses each POI's POST-construction geometry, since Ceng is defined for "the post-construction geometry."
UNINTERRUPTED_CLASSES = {"freeway", "ramp"}

fc_norm = poi["functional_class_post"].astype(str).str.strip().str.lower()
is_uninterrupted = fc_norm.isin(UNINTERRUPTED_CLASSES)
sigma = np.where(is_uninterrupted, 0.1427, 0.3907)
beta_hers = np.where(is_uninterrupted, 0.2092, 0.18378)

# No curvature/pavement-roughness data available -> assume straight road, good pavement
# (VCURVE = VROUGH = VSPLIM), per the doc's stated default for this case.
VSPLIM = poi["speed_limit_post"].astype(float)
VCURVE = VSPLIM
VROUGH = VSPLIM

base_ffs = (np.exp(sigma**2 / 2) * (VCURVE**(-1/beta_hers) + VROUGH**(-1/beta_hers) + VSPLIM**(-1/beta_hers)))**(-beta_hers)

# Lane-width penalty (12ft=0, 11ft=1.5mph, 10ft=4mph, <10ft=6.5mph)
lane_width = poi["lane_width_post"].astype(float)
fw = np.select([lane_width >= 12, lane_width >= 11, lane_width >= 10], [0, 1.5, 4], default=6.5)
ffs_lw = base_ffs - fw

# Signal adjustment (doc formula, NOT the buggy xlsx one - see markdown above)
nsig = poi["signals_per_mile_post"].astype(float)
zvdsig = np.where(nsig == 0, 0, 0.0687 * (1 - np.exp(-nsig / 24.4)))
ffs_sig = 1 / (1/ffs_lw + zvdsig)

# Stop-sign adjustment
nstp = poi["stop_signs_per_mile_post"].astype(float)
zvdstp = np.where(nstp == 0, 0, nstp * (1.9 + 0.067 * ffs_sig))
ffs_final = 1 / (1/ffs_sig + zvdstp / 1000)

# Speed at capacity (HCM density = 35 pc/mi/ln) -> flow per lane -> Ceng
v_cap = ffs_final * 0.9
s_cap = v_cap * 35
poi["engineering_capacity_ceng"] = s_cap * poi["through_lanes_post"].astype(float) * f_hv

poi[["functional_class_post", "speed_limit_post", "through_lanes_post", "engineering_capacity_ceng"]].head(10)

,functional_class_post,speed_limit_post,through_lanes_post,engineering_capacity_ceng
poi_id,,,,
6788,freeway,55.0,4.0,4659.911495
6789,principal arterial,55.0,2.0,2652.465768
9834,principal arterial,NaN,NaN,NaN
9836,principal arterial,NaN,NaN,NaN
9860,principal arterial,NaN,NaN,NaN
9873,principal arterial,NaN,NaN,NaN
9884,freeway,NaN,NaN,NaN
10026,minor arterial,55.0,2.0,2562.690768
10027,minor arterial,55.0,2.0,2652.465768


In [11]:
# Exponent X = sum(beta x variable), multiplier M = e^X, PostCapacity = Ceng x M
X = (
    betas["bLM"] * poi["d_lane_miles"]
    + betas["bL"] * poi["d_through_lanes"]
    + betas["bLT"] * poi["d_left_turn_lanes"]
    + betas["bRT"] * poi["d_right_turn_lanes"]
    + betas["bW"] * poi["lane_width_improved"]
    + betas["bE"] * poi["express_lane_added"]
    + betas["bS"] * poi["shoulder_widened"]
    + betas["bF"] * poi["functional_class_improved"]
    + betas["bA"] * poi["d_ln_aadt"]
    + betas["bP"] * 0  # no regional population-growth data joined in yet
    + betas["bY"] * 0  # no regional income-growth data joined in yet
    + betas["bSL"] * poi["speed_limit_increased"]
    + betas["bSIG"] * poi["signals_per_mile_post"]
    + betas["bAT"] * poi["area_type_level"]
)

poi["capacity_multiplier"] = np.exp(X)
poi["peak_hour_capacity"] = poi["engineering_capacity_ceng"] * poi["capacity_multiplier"]

# Merge the POI-level results back onto every observation row for that POI
_poi_out = poi[["engineering_capacity_ceng", "capacity_multiplier", "peak_hour_capacity"]].reset_index()
master_df = master_df.drop(columns=["engineering_capacity_ceng", "capacity_multiplier", "peak_hour_capacity"], errors="ignore")
master_df = master_df.merge(_poi_out, on="poi_id", how="left")

master_df[[
    "poi_id", "functional_class", "before_or_after",
    "engineering_capacity_ceng", "capacity_multiplier", "peak_hour_capacity",
]].head(10)

,poi_id,functional_class,before_or_after,engineering_capacity_ceng,capacity_multiplier,peak_hour_capacity
0,40974,minor arterial,before,2652.465768,0.807688,2142.363688
1,40974,minor arterial,before,2652.465768,0.807688,2142.363688
2,40974,minor arterial,before,2652.465768,0.807688,2142.363688
3,40974,minor arterial,after,2652.465768,0.807688,2142.363688
4,40974,minor arterial,after,2652.465768,0.807688,2142.363688
5,40974,minor arterial,after,2652.465768,0.807688,2142.363688
6,40974,minor arterial,after,2652.465768,0.807688,2142.363688
7,40974,minor arterial,after,2652.465768,0.807688,2142.363688
8,40974,minor arterial,after,2652.465768,0.807688,2142.363688
9,40974,minor arterial,after,2652.465768,0.807688,2142.363688


## 6. Spot-check the merged data

Displays a narrowed view of just the columns most relevant for spot-checking, the join keys, dates, before/after labels, and a few geometric attributes (lane counts, shoulder width), to visually verify the merges worked correctly before writing anything out.

In [12]:
master_df[[
    "project_id", "poi_id", "observation_date", "date_open_actual",
    "before_or_after", "before_or_after_binary", "years_from_open",
    "through_lanes", "right_turn_lanes", "shoulder_width",
    "functional_class", "k_factor", "d_factor", "peak_hour_capacity",
    "peak_hour_volume",
]]

,project_id,poi_id,observation_date,date_open_actual,before_or_after,before_or_after_binary,years_from_open,through_lanes,right_turn_lanes,shoulder_width,functional_class,k_factor,d_factor,peak_hour_capacity,peak_hour_volume
0,F-M-0436,40974,2011-01-01,2017-04-01,before,0,-6.0,2.0,0.0,7.0,minor arterial,0.10,0.55,2142.363688,918.500
1,F-M-0436,40974,2016-01-01,2017-04-01,before,0,-1.0,2.0,0.0,7.0,minor arterial,0.10,0.55,2142.363688,963.490
2,F-M-0436,40974,2017-01-01,2017-04-01,before,0,0.0,2.0,0.0,7.0,minor arterial,0.10,0.55,2142.363688,981.420
3,F-M-0436,40974,2018-01-01,2017-04-01,after,1,1.0,2.0,1.0,9.0,minor arterial,0.10,0.55,2142.363688,986.535
4,F-M-0436,40974,2019-01-01,2017-04-01,after,1,2.0,2.0,1.0,9.0,minor arterial,0.10,0.55,2142.363688,836.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1959,TA-M352,10707,2021-01-01,NaT,after,1,NaN,NaN,NaN,NaN,principal arterial,0.12,0.60,<NA>,3485.376
1960,TA-M352,10707,2022-01-01,NaT,after,1,NaN,NaN,NaN,NaN,principal arterial,0.12,0.60,<NA>,3414.024
1961,TA-M352,10707,2023-01-01,NaT,after,1,NaN,NaN,NaN,NaN,principal arterial,0.12,0.60,<NA>,3673.944
1962,TA-M352,10707,2024-01-01,NaT,after,1,NaN,NaN,NaN,NaN,principal arterial,0.12,0.60,<NA>,4224.744


## 7. Write out the combined CSV

Saves `master_df` to `combined_output.csv` inside `DATA_ROOT` (the same folder your project subfolders live in, alongside `validation_report.txt`) and confirms how many rows/columns were written. This file is the input for the quantile regression model-fitting notebook.

In [13]:
output_df = master_df.drop(columns=["engineering_capacity_ceng", "capacity_multiplier"], errors="ignore")

output_path = os.path.join(DATA_ROOT, "combined_output.csv")
output_df.to_csv(output_path, index=False)
print(f"Wrote {len(output_df)} rows and {len(output_df.columns)} columns to {output_path}")

Wrote 1964 rows and 38 columns to D:\Documents\Code\forecastcards_ba_data\data\combined_output.csv
